[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daleas0120/sdl-workshops/blob/main/statistical_doe/DOE.ipynb)

# Statistical Design of Experiments on a Color Mixing Self-Driving Lab

This notebook shows you how to design, do, and analyze a factorial experiment.

The equipment is an Opentrons OT-2 liquid handler. It mixes red, yellow, and
blue dye. A light sensor with 8 channels then measures the mixture. The robot is
real, and you can use it through the internet.

You get 26 experiments. You cannot get more. This limit controls every decision
in this notebook.

## How to use this notebook

This notebook does not run from the first cell to the last cell. It stops at
each task, and you write the missing code. This is deliberate.

A task cell contains blanks:

```python
# ── TASK 1 ──────────────────────────────────────────────────────────
levels = {"R": [___, ___]}   # replace each ___
```

Each `___` is a blank. If you do not replace a blank, the cell stops with the
message `NameError: name '___' is not defined`.

Each task has three cells:

| Cell | Function |
|---|---|
| 📋 Brief | Tells you what to write, and why it is necessary |
| ✏️ Your code | Contains the blanks |
| ✅ Check | Tells you if your code is correct |

Each brief starts with a **What you already have** table. It lists every
variable that the task needs, its type, and what is inside it. Each blank also
has a comment that gives its type.

Two more tools help you:

- **Part 1b** shows every Python and pandas pattern that the tasks use, with
  output that you can run. Go back to it when a blank is not clear.
- **`whats_in_memory()`** prints each variable that you have made, with its type
  and its size. Run it in any empty cell, at any time.

Each brief also has a `▸ Hints` block. Open it if you cannot continue. The
Solutions appendix at the end has the full answers. Do the task first. You learn
from the blanks, and not from the answers.

Write your code in simulation first. This notebook contains a simulator of the
equipment (`USE_HARDWARE = False`). The simulator is free, and you can use it as
many times as necessary. Change to the robot only after all the checks pass.

---

## Part 0 · Background

### Self-driving laboratories

Scientists find new materials slowly. Most laboratory work is manual. A person
holds the pipette, mixes the liquids, and writes down the data. Scientists found
many important materials by accident.

A self-driving laboratory closes the loop:

1. The equipment does an experiment.
2. The sensors measure the result.
3. An algorithm selects the next experiment.
4. The loop starts again.

The scientist then does the work that needs a person: to make hypotheses, to
design campaigns, and to interpret data.

<p align="center">
<img src="https://github.com/sparks-baird/self-driving-lab-demo/blob/main/notebooks/map-diagram-2.png?raw=true" width="360">
</p>

This notebook uses all four steps of the loop:

1. **Send commands to the equipment.** You send three dye volumes to an API.
2. **Read the sensor data.** You get 8 light intensity values.
3. **Select the next experiment.** You select all the experiments before you
   start.
4. **Model the result.** You calculate effects, do an ANOVA, and plot a response
   surface.

Step 3 is different in this notebook. [`BO.ipynb`](./BO.ipynb) in this folder
uses a Bayesian optimizer. That algorithm selects each experiment after it
examines all the earlier data, and it searches for one best recipe. This
notebook selects all 26 experiments first. The goal is different. You must find
how the full system operates.

### The equipment: a color mixing OT-2

<p align="center">
<img src="https://github.com/sparks-baird/self-driving-lab-demo/blob/main/notebooks/clslab-light.gif?raw=1" width="380">
<br><sup>The same loop in a light mixing form. The color mixing OT-2 has a
pipette and dye in place of the LED.</sup>
</p>

The [Acceleration Consortium](https://acceleration.utoronto.ca/) operates this
equipment. You send jobs to it through the internet.

| Component | Function |
|---|---|
| **Opentrons OT-2** | A robot that moves a pipette. It takes dye from three containers and puts it in a well. |
| **Red, yellow, and blue dye** | You select the volume of each dye, from **1 µL to 299 µL**. |
| **Light sensor** | Measures the light in **8 channels**, near 410, 440, 470, 510, 550, 583, 620, and 670 nm. |
| **Gradio API** | Receives `(student_id, r_vol, y_vol, b_vol)`, puts the job in a queue, and returns the 8 sensor values. |

One job takes about **two minutes**. The robot pipettes, mixes, measures, and
reports. Twenty-six jobs take most of one hour. This is why your time slot is
one hour long.

#### Three limits that control the design

1. **Budget: 26 experiments.** You get 25 submissions, and one spare submission
   to test your code.
2. **Volume: use 10 µL or more.** The API accepts 1 µL. But below 10 µL the
   pipette error is large. This error can be larger than the effects that you
   want to measure.
3. **Stray light that changes with the well position.** Ambient light in the
   OT-2 goes into the sensor. The platform documentation gives a value of about
   **10%** of the response. This light is not random noise. It stays the same
   for the same well. Therefore it is a **systematic error**, and replication
   does not remove it. You must find this error in your data and measure it. Do
   not accept the value of 10% without a test.

### Why 26 experiments is a small number

You want to know how each dye changes the color. The usual method is **one
factor at a time** (OFAT). You keep yellow and blue constant and change red.
Then you keep red and blue constant and change yellow.

OFAT has one large problem. The effect of red can change with the quantity of
blue, because the two dyes absorb the same light. OFAT tests red at one level of
blue only. Therefore OFAT cannot find this behavior. OFAT measures **main
effects**, and it is blind to **interactions**.

A **factorial design** changes all the factors together, in a balanced pattern.
Each experiment then gives data about each effect.

| Method | What it measures | Experiments for 3 factors at 2 levels |
|---|---|---|
| OFAT | 3 main effects | about 7 |
| **2³ full factorial** | 3 main effects, 3 two-way interactions, 1 three-way interaction | **8** |

The two methods use almost the same number of experiments. The factorial design
gives more data. The difference increases when you add more factors.

#### Terms that you need

- **Factor** — an input that you control. Here: the `R`, `Y`, and `B` volumes.
- **Level** — a value of a factor. This design uses a **low** level and a
  **high** level.
- **Response** — the number that you measure and model. Here it comes from the 8
  sensor channels.
- **Coded units** — a scale where the low level is **−1** and the high level is
  **+1**. On this scale you can compare the effects directly. Each interaction
  column is also a product of two other columns.
- **Main effect** — the mean change in the response when a factor moves from −1
  to +1.
- **Interaction** — the quantity by which the effect of one factor changes with
  the level of a different factor.
- **Replication** — the same recipe as two different jobs. Replication measures
  **pure error**, the difference between two identical experiments. To
  measure one well two times is *repetition*, and not replication. Repetition
  measures the sensor noise only.
- **Center point** — an experiment at the middle level of all the factors. A
  center point finds **curvature**. A design with two levels cannot find
  curvature.
- **Randomization** — a random sequence for the experiments. Randomization
  spreads the time-related changes across all the factors. Temperature drift,
  lamp drift, and well position are examples of time-related changes.

#### Your budget of 26 experiments

| Block | Experiments | What it gives you |
|---|---:|---|
| 2³ factorial corners, two replicates | 16 | all main effects, all interactions, and pure error |
| Center points | 4 | a curvature test, and more pure error |
| Validation points | 5 | an independent test of the predictions from your model |
| Pipeline test | 1 | proof that your code operates correctly |
| **Total** | **26** | |

---

## Part 1 · Setup

Run the next cells. Do not change them. They install the software, set the
constants, and give you the interface to the equipment and to the simulator.

These cells contain no blanks. But read the code. Look at `run_experiment`,
because your loop in Task 3 calls this function.

In [ ]:
# Software. You need `gradio_client` for the real robot only.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q gradio_client statsmodels plotly

In [ ]:
import itertools
import json
import os
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# ── Session configuration ───────────────────────────────────────────────────
USE_HARDWARE = False          # change to True in your OT-2 time slot only
STUDENT_ID = "test1"          # your student or team ID. "debug" uses the test endpoint.
SEED = 403                    # the assignment specifies 403 or 1003

# ── Equipment constants ─────────────────────────────────────────────────────
CHANNELS = ["ch410", "ch440", "ch470", "ch510", "ch550", "ch583", "ch620", "ch670"]
WAVELENGTHS = np.array([410, 440, 470, 510, 550, 583, 620, 670], dtype=float)

# The reading with no dye in the cell. On the robot, measure this one time and
# write the values here. This is I0 in the Beer-Lambert equation, and every
# calculation below uses it.
REFERENCE_SPECTRUM = np.array([1200, 1600, 1900, 2400, 2600, 2300, 2000, 1500], dtype=float)

# ── Limits from the assignment ──────────────────────────────────────────────
QUOTA = 26              # total submissions. You cannot get more.
VOL_MIN, VOL_MAX = 1.0, 299.0     # the range that the API accepts, for each dye
VOL_RECOMMENDED_MIN = 10.0        # below this volume, the pipette error is large
TOTAL_VOL_MAX = 300.0             # keep the liquid in the well

RESULTS_CSV = "doe_results.csv" if USE_HARDWARE else "doe_results_sim.csv"

rng = np.random.default_rng(SEED)

print(f"mode        : {'LIVE HARDWARE' if USE_HARDWARE else 'simulation'}")
print(f"results file: {RESULTS_CSV}")
print(f"budget      : {QUOTA} experiments")

In [ ]:
# ── Interface to the robot ──────────────────────────────────────────────────
# This is the same Gradio endpoint and the same error control as BO.ipynb in
# this folder.

_client = None


def _get_client():
    global _client
    if _client is None:
        from gradio_client import Client
        _client = Client("https://accelerationconsortium-ot-2-lcm.hf.space/")
    return _client


def submit_experiment(student_id, r_vol, y_vol, b_vol):
    """Send one color mixing job to the OT-2 and wait for the sensor data.

    Returns (result_dict, killed). The keys of result_dict["Sensor Data"] are
    the names in CHANNELS. Raises SystemExit if an administrator must help.
    """
    killed = False
    api_endpoint = "/debug" if student_id == "debug" else "/submit"

    job = _get_client().submit(student_id, r_vol, y_vol, b_vol, api_name=api_endpoint)
    result = job.result()

    msg = result.get("Message", "")
    if msg == "Out of tips. Please contact adminstrator to restock.":
        raise SystemExit("Queue terminated by administrator. REASON: Out of tips")
    elif msg == "Plate full. Please contact adminstrator to replace.":
        raise SystemExit("Queue terminated by administrator. REASON: Plate full")
    elif msg == "Queue killed. Please contact adminstrator to restart program.":
        killed = True

    return result, killed

In [ ]:
# ── Simulator ───────────────────────────────────────────────────────────────
# A Beer-Lambert model of the same equipment. Use it to write your code at no
# cost. It has the two error sources of the real equipment: a stray light offset
# that changes with the well position (systematic), and sensor noise (random).
#
# ASSUMPTION: the sensor measures through a cell with a constant path length.
# Therefore the absorbance changes with the dye concentration, and not with the
# absolute volume. One result of this assumption: (30, 30, 30) and (90, 90, 90)
# give the same color. The real equipment can be different. Your design tests
# this in Task 8.


def _band(peak_nm, width_nm, amplitude):
    return amplitude * np.exp(-0.5 * ((WAVELENGTHS - peak_nm) / width_nm) ** 2)


# The absorptivity of each dye in the 8 sensor channels, in absorbance units for
# each unit of volume fraction, for each cm of path length. These shapes are
# typical of common food dyes: allura red (505 nm), tartrazine (428 nm), and
# brilliant blue (630 nm).
DYE_ABSORPTIVITY = {
    "R": _band(505.0, 58.0, 1.15) + _band(560.0, 40.0, 0.35),
    "Y": _band(428.0, 45.0, 1.00) + _band(470.0, 35.0, 0.20),
    "B": _band(630.0, 60.0, 1.35) + _band(590.0, 45.0, 0.30),
}
PATHLENGTH_CM = 1.0

_STRAY_MIN, _STRAY_MAX = 0.004, 0.009   # ambient light, as a fraction of full scale
_READ_NOISE_FRAC = 0.002                # sensor noise, in the same units
_sim_rng = np.random.default_rng(SEED)


def plate_position(well_index):
    """Change an experiment number into a position on a 96-well plate.

    The robot fills the plate one row at a time. Returns (row_letter, column).
    """
    i = (int(well_index) - 1) % 96
    row, col = i // 12, i % 12
    return "ABCDEFGH"[row], col + 1


def _ideal_spectrum(r_vol, y_vol, b_vol):
    total = float(r_vol) + float(y_vol) + float(b_vol)
    fractions = {"R": r_vol / total, "Y": y_vol / total, "B": b_vol / total}
    absorbance = PATHLENGTH_CM * sum(DYE_ABSORPTIVITY[d] * fractions[d] for d in fractions)
    return REFERENCE_SPECTRUM * 10.0 ** (-absorbance)


def simulate_ot2(r_vol, y_vol, b_vol, well_index=1):
    """Replace submit_experiment() and return a dictionary of the same shape."""
    _row, col = plate_position(well_index)
    # Systematic error: the stray light increases across the plate columns. The
    # same well always gives the same offset. Replication cannot remove it.
    stray = (_STRAY_MIN + (_STRAY_MAX - _STRAY_MIN) * (col - 1) / 11.0) * REFERENCE_SPECTRUM
    # Random error: sensor noise with a mean of zero.
    noise = _sim_rng.normal(0.0, _READ_NOISE_FRAC * REFERENCE_SPECTRUM)

    counts = np.clip(np.round(_ideal_spectrum(r_vol, y_vol, b_vol) + stray + noise), 1, 65535)
    return {"Message": "Success", "Sensor Data": {c: float(v) for c, v in zip(CHANNELS, counts)}}

In [ ]:
# ── The function that your loop calls ───────────────────────────────────────


def run_experiment(r_vol, y_vol, b_vol, well_index=1):
    """Do one mix on the OT-2 or on the simulator.

    Returns (spectrum, killed). spectrum is a numpy array of 8 counts, in the
    sequence of CHANNELS.
    """
    if USE_HARDWARE:
        result, killed = submit_experiment(STUDENT_ID, float(r_vol), float(y_vol), float(b_vol))
    else:
        result, killed = simulate_ot2(r_vol, y_vol, b_vol, well_index), False

    spectrum = np.array([result["Sensor Data"][c] for c in CHANNELS], dtype=float)
    return spectrum, killed


def append_result(row, path=RESULTS_CSV):
    """Write one completed experiment to the CSV file immediately.

    Write to disk immediately. If the queue stops at experiment 19, you keep 18
    measurements. If the data is only in memory, you lose the full hour.
    """
    df = pd.DataFrame([row])
    df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)


def load_results(path=RESULTS_CSV):
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()

In [ ]:
# ── Color calculation and design validation ─────────────────────────────────
from IPython.display import HTML, display

# The CIE 1931 2-degree color matching functions, at the 8 sensor channels.
_CMF = np.array([
    [0.1344, 0.0040, 0.6456], [0.3483, 0.0230, 1.7471], [0.1954, 0.0910, 1.2876],
    [0.0093, 0.5030, 0.1582], [0.4334, 0.9950, 0.0087], [0.9824, 0.8386, 0.0015],
    [1.0622, 0.6310, 0.0008], [0.2835, 0.1070, 0.0000],
])
_WHITE = _CMF.sum(axis=0) / _CMF[:, 1].sum()
_D65 = np.array([0.95047, 1.00000, 1.08883])
_XYZ_TO_RGB = np.array([
    [3.2406, -1.5372, -0.4986], [-0.9689, 1.8758, 0.0415], [0.0557, -0.2040, 1.0570]])


def spectrum_to_hex(spectrum, reference=REFERENCE_SPECTRUM):
    """Change an 8-channel spectrum into an approximate sRGB color."""
    transmittance = np.clip(np.asarray(spectrum, dtype=float) / reference, 0.0, 1.0)
    xyz = (transmittance[:, None] * _CMF).sum(axis=0) / _CMF[:, 1].sum()
    xyz = xyz / _WHITE * _D65                      # change equal-energy white to D65
    linear = np.clip(_XYZ_TO_RGB @ xyz, 0.0, 1.0)
    srgb = np.where(linear <= 0.0031308, 12.92 * linear, 1.055 * linear ** (1 / 2.4) - 0.055)
    return "#%02x%02x%02x" % tuple(int(round(v * 255)) for v in np.clip(srgb, 0, 1))


def show_swatches(labels, hexes, size=54):
    """Show a row of color squares with labels."""
    cells = "".join(
        f'<div style="text-align:center;margin:4px 6px;font:11px sans-serif">'
        f'<div style="width:{size}px;height:{size}px;background:{h};'
        f'border:1px solid #8888;border-radius:6px"></div>{l}</div>'
        for l, h in zip(labels, hexes)
    )
    display(HTML(f'<div style="display:flex;flex-wrap:wrap">{cells}</div>'))


def validate_design(design):
    """Do the 'test library validity' step from the assignment.

    Returns a list of the problems. Run this before you use your experiments.
    """
    problems = []
    n = len(design)
    if n > QUOTA:
        problems.append(f"the design has {n} experiments, but the quota is {QUOTA}")
    for col in ("R", "Y", "B"):
        if col not in design.columns:
            problems.append(f"the column '{col}' is not present")
    if problems:
        return problems
    vols = design[["R", "Y", "B"]].to_numpy(dtype=float)
    if (vols < VOL_MIN).any() or (vols > VOL_MAX).any():
        problems.append(f"a volume is outside the range {VOL_MIN} to {VOL_MAX} uL")
    if (vols < VOL_RECOMMENDED_MIN).any():
        problems.append(f"a volume is below the {VOL_RECOMMENDED_MIN} uL accuracy limit")
    if (vols.sum(axis=1) > TOTAL_VOL_MAX).any():
        problems.append(f"a total volume is more than {TOTAL_VOL_MAX} uL")
    return problems


# ── Support for the check cells ─────────────────────────────────────────────
class _Checker:
    def __init__(self, name):
        self.name, self.failures = name, []

    def that(self, condition, passed_msg, failed_msg):
        if condition:
            print(f"  ✅ {passed_msg}")
        else:
            print(f"  ❌ {failed_msg}")
            self.failures.append(failed_msg)
        return self

    def done(self):
        if self.failures:
            raise AssertionError(
                f"{self.name}: {len(self.failures)} check(s) failed. Read the messages above.")
        print(f"\U0001f389 {self.name}: all checks passed")


def checker(name):
    print(f"{name}")
    return _Checker(name)


# ── What is in memory? ──────────────────────────────────────────────────────
_TRACKED = [
    "CHANNELS", "WAVELENGTHS", "REFERENCE_SPECTRUM", "DYE_ABSORPTIVITY",
    "PATHLENGTH_CM", "QUOTA", "SEED", "LOW", "HIGH", "CENTER", "half_range",
    "design", "results", "factorial", "centers", "validation", "train",
    "A", "effects", "RESPONSE", "FACTORS", "model", "anova_table",
    "predicted", "measured", "E_fitted",
]


def _describe(v):
    """Return (type, size, contents) as three short strings."""
    if isinstance(v, pd.DataFrame):
        return ("DataFrame", f"{v.shape[0]} rows x {v.shape[1]} cols",
                ", ".join(list(v.columns)[:6]) + (" ..." if v.shape[1] > 6 else ""))
    if isinstance(v, pd.Series):
        return ("Series", f"{len(v)} values", f"dtype {v.dtype}")
    if isinstance(v, np.ndarray):
        return ("numpy array", f"shape {v.shape}", f"dtype {v.dtype}")
    if isinstance(v, dict):
        return ("dict", f"{len(v)} keys", ", ".join(str(k) for k in list(v)[:6]))
    if isinstance(v, (list, tuple)):
        kind = "list" if isinstance(v, list) else "tuple"
        return (kind, f"{len(v)} items", str(v[:4])[:44])
    if isinstance(v, (int, float, str, bool)):
        return (type(v).__name__, "1 value", repr(v)[:44])
    if hasattr(v, "params") and hasattr(v, "resid"):
        return ("model result", f"{int(v.nobs)} experiments",
                f"{len(v.params)} terms. Use .fittedvalues and .resid")
    return (type(v).__name__, "", "")


def whats_in_memory(*names):
    """Print each variable that this notebook made, with its type and size.

    Run this cell at any time. It answers the question "what do I already have,
    and what is inside it?".
    """
    names = names or _TRACKED
    g = globals()
    rows = [(n,) + _describe(g[n]) for n in names if n in g]
    if not rows:
        print("Nothing yet. Run the cells above first.")
        return
    w = [max(len(r[i]) for r in rows) for i in range(4)]
    head = ("name", "type", "size", "contents")
    print(f"  {head[0]:<{w[0]}}  {head[1]:<{w[1]}}  {head[2]:<{w[2]}}  {head[3]}")
    print("  " + "-" * (w[0] + w[1] + w[2] + w[3] + 6))
    for r in rows:
        print(f"  {r[0]:<{w[0]}}  {r[1]:<{w[1]}}  {r[2]:<{w[2]}}  {r[3]}")


print("Setup complete.")

---

## Part 1b · The Python patterns that the tasks use

The subject of this notebook is design of experiments, and not Python. This
section shows every Python and pandas pattern that the tasks need. Run the next
cell and read the output. Come back to it when a blank is not clear.

Three container types come up again and again:

| Type | Looks like | You can change it | Used here for |
|---|---|---|---|
| **list** | `[30.0, 90.0]` | yes | a group of values that you give to a function |
| **tuple** | `(30.0, 90.0)` | no | one row of data, or two values back from a function |
| **dict** | `{"R": 30.0}` | yes | a name for each value, such as one experiment record |

A function that returns **two values** gives you a tuple. You unpack it with two
names on the left:

```python
spectrum, killed = run_experiment(30, 30, 30, well_index=2)
```

In [ ]:
# Run this cell and read the output. Each block is one pattern that a task needs.
demo = pd.DataFrame({
    "block": ["factorial", "center", "factorial"],
    "R": [30.0, 60.0, 90.0],
    "Y": [30.0, 60.0, 30.0],
})

print("1. A DataFrame is a table. Each column has a name.")
print(demo)
print(f"   demo.shape = {demo.shape}  ->  {demo.shape[0]} rows, {demo.shape[1]} columns")
print(f"   demo.columns = {list(demo.columns)}\n")

print("2. ONE column gives a Series. Write demo['R'] or demo.R")
print("  ", demo["R"].to_list(), "\n")

print("3. TWO OR MORE columns need a LIST inside the brackets: demo[['R', 'Y']]")
print(demo[["R", "Y"]], "\n")

print("4. To keep some rows, put a True/False Series inside the brackets.")
print("   demo.block == 'factorial' gives:", demo.block.eq("factorial").to_list())
print(demo[demo.block == "factorial"], "\n")

print("5. To add a column, give a value to a new name.")
demo["total"] = demo.R + demo.Y      # pandas adds row by row
print(demo, "\n")

print("6. itertuples() gives one object for each row. Read a column with row.NAME")
for row in demo.itertuples():
    print(f"   row.R = {row.R:5.1f}   row.Y = {row.Y:5.1f}   row.block = {row.block}")
print()

print("7. to_numpy() changes a DataFrame into a numpy array.")
arr = demo[["R", "Y"]].to_numpy()
print(f"   arr has shape {arr.shape}: {arr.shape[0]} rows and {arr.shape[1]} columns")
print("   arr.sum(axis=0) adds DOWN the rows    ->", arr.sum(axis=0))
print("   arr.sum(axis=1) adds ACROSS each row  ->", arr.sum(axis=1))
print("   Remember: axis=1 gives one number for each row.\n")

print("8. numpy arrays do arithmetic on every element at the same time.")
print("   np.array([1, 2, 3]) * 10   ->", np.array([1, 2, 3]) * 10)
print("   np.array([1, 2, 3]) + np.array([10, 20, 30]) ->",
      np.array([1, 2, 3]) + np.array([10, 20, 30]), "\n")

print("9. A dictionary maps a key to a value.")
example = {"R": 30.0, "Y": 60.0}
print("   example['R'] ->", example["R"])
print("   for key in example: ...  goes through the KEYS ->", list(example))

In [ ]:
# Run whats_in_memory() at any time. It shows each variable that this notebook
# has made, its type, its size, and what is inside it.
whats_in_memory()

---

## Part 2 · Design the experiment

### 📋 Task 1 — Make the design matrix

**What you already have:**

| Name | Type | What it holds |
|---|---|---|
| `SEED` | int | `403`. Give it to `random_state` to get the same sequence every time. |
| `QUOTA` | int | `26`. The total number of submissions. |
| `itertools` | module | `itertools.product(a_list, repeat=n)` makes combinations. |
| `pd`, `np` | modules | pandas and numpy. |

**What you must make:** a variable named `design`. It is a pandas DataFrame with
25 rows. Experiment 26 is the pipeline test in Part 3. The DataFrame must have
these columns:

| Column | Content |
|---|---|
| `run_order` | 1 to 25. The sequence of the submissions. |
| `block` | `"factorial"`, `"center"`, or `"validation"` |
| `R`, `Y`, `B` | the dye volumes, in µL |
| `well_index` | the well on the plate for this experiment |

Do these four steps:

1. **Select the two levels.** `LOW` and `HIGH` must be 10 µL or more. The
   largest total (`3 × HIGH`) must be 300 µL or less. `CENTER` is the middle
   value.
2. **Make all 8 corners two times.** Three factors at two levels give
   `2 × 2 × 2` combinations. `itertools.product` makes these combinations for
   you.
3. **Randomize the sequence.** Use `random_state=SEED`, and your campaign is
   then repeatable. A slow drift during the hour can look like a dye effect.
   Randomization prevents this.
4. **Give a well index to each experiment.** The robot fills the plate in
   sequence. Therefore `well_index = run_order + 1` is correct here, because the
   pipeline test uses well 1. Step 3 makes the well position independent of your
   factors. Because of this, you can find a position effect later. Without
   randomization, a position effect changes your effect estimates.

<details>
<summary>▸ Hints</summary>

- `3 * HIGH <= 300` gives the largest possible value of `HIGH`. Use a smaller
  value.
- `itertools.product([LOW, HIGH], repeat=3)` gives the 8 corner combinations.
- To replicate a corner, submit the same recipe as a second job. Therefore make
  the list of corners two times.
- `df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)` mixes the rows.
- Add `run_order` after you mix the rows.

</details>

In [ ]:
# ── TASK 1 ──────────────────────────────────────────────────────────────────
LOW = ___          # float. The low level in uL. It must be 10.0 or more.
HIGH = ___         # float. The high level in uL. 3 * HIGH must be 300.0 or less.
CENTER = ___       # float. The middle value between LOW and HIGH.

N_REPLICATES = ___     # int. How many times to do the full set of 8 corners.
N_CENTER = ___         # int. How many center points.

# 1. The 2^3 factorial corners, with replicates.
#    itertools.product(a_list, repeat=n) gives every combination of n items
#    from a_list. For example, product([1, 2], repeat=2) gives
#    (1,1), (1,2), (2,1), (2,2).
#    First blank : a LIST that holds the two levels.
#    Second blank: an int, the number of factors.
corner_rows = []
for _replicate in range(N_REPLICATES):
    for r_vol, y_vol, b_vol in itertools.product(___, repeat=___):
        corner_rows.append({"block": "factorial", "R": r_vol, "Y": y_vol, "B": b_vol})

# 2. The center points. Each blank is a float: the volume of that dye.
center_rows = [{"block": "center", "R": ___, "Y": ___, "B": ___} for _ in range(N_CENTER)]

# 3. The validation points (given). These points are inside the design space.
#    Each one is a fraction of the distance from LOW to HIGH. Therefore they
#    agree with the levels that you selected.
VALIDATION_FRACTIONS = [
    (0.25, 0.75, 0.50), (0.75, 0.25, 0.00), (0.00, 0.50, 1.00),
    (1.00, 0.00, 0.25), (0.50, 1.00, 0.75),
]
validation_rows = [
    {"block": "validation",
     "R": LOW + fr * (HIGH - LOW), "Y": LOW + fy * (HIGH - LOW), "B": LOW + fb * (HIGH - LOW)}
    for fr, fy, fb in VALIDATION_FRACTIONS
]

# 4. Put the rows together, mix them, and give a number to each experiment.
design = pd.DataFrame(corner_rows + center_rows + validation_rows)
# The blank is an int. Use SEED, and the sequence is then the same every time.
design = design.sample(frac=1.0, random_state=___).reset_index(drop=True)
design.insert(0, "run_order", np.arange(1, len(design) + 1))
design["well_index"] = design["run_order"] + 1   # the pipeline test uses well 1

design

In [ ]:
# ✅ CHECK — Task 1
c = checker("Task 1 - design matrix")
corners = design[design.block == "factorial"]
c.that(len(design) == 25,
       f"25 experiments in the design, and 1 pipeline test. Total = {QUOTA}.",
       f"the design has {len(design)} rows. It needs 25: "
       "16 corners, 4 centers, and 5 validation points.")
c.that(len(design) + 1 <= QUOTA, "the design is inside the quota",
       f"{len(design)} experiments and 1 test are more than the quota of {QUOTA}")
c.that(len(corners.groupby(['R', 'Y', 'B'])) == 8,
       "all 8 factorial corners are present",
       f"the design needs 8 different corners. It has "
       f"{len(corners.groupby(['R', 'Y', 'B']))}.")
c.that(len(corners) == 16, "each corner has two replicates",
       f"the design needs 16 factorial experiments. It has {len(corners)}.")
c.that(set(corners[['R', 'Y', 'B']].to_numpy().ravel()) == {LOW, HIGH},
       "the corners use the low level and the high level only",
       "a corner has a volume that is not LOW and not HIGH")
c.that(abs(CENTER - (LOW + HIGH) / 2) < 1e-9,
       "CENTER is the middle value between LOW and HIGH",
       f"CENTER must be {(LOW + HIGH) / 2}. It is {CENTER}.")
c.that(not validate_design(design),
       "the design agrees with the limits of the equipment",
       "problem with the limits: " + "; ".join(validate_design(design)))
block_groups = int((design.block != design.block.shift()).sum())
c.that(block_groups >= 6,
       f"the sequence is random. The blocks are mixed together "
       f"({block_groups} groups).",
       f"the blocks are still in {block_groups} groups, so the sequence is not "
       "random. Mix the rows with .sample(frac=1.0, random_state=SEED) before "
       "you add the run_order column.")
c.that(design.well_index.nunique() == len(design),
       "each experiment has a different well",
       "two experiments have the same well_index")
c.done()

Look at your design. The `factorial` experiments are not together in time, and
the center points are also not together. Randomization does this.

The lamp can drift. The room can become warmer. The stray light can change
across the plate. Randomization sends these changes to all your factor levels in
equal quantities.

The changes then make your estimate of the error larger. That result is correct.
The changes do not move one effect only. That result is incorrect.

---

### 📋 Task 2 — Calculate the expected color

The assignment tells you to predict the color for a set of RYB volumes. Then you
measure the color and compare. To predict a color, you need a physical model.
For dyes in a liquid, this model is the **Beer-Lambert law**:

$$A_\lambda \;=\; \varepsilon_\lambda \, c \, \ell$$

In this equation, $A$ is the absorbance at the wavelength $\lambda$. The
absorptivity is $\varepsilon_\lambda$, the concentration is $c$, and the path
length is $\ell$.

Two properties of this law are important:

- **Absorbances add together.** For a mixture,
  $A_\lambda = \ell \sum_d \varepsilon_{d,\lambda} c_d$.
- **Intensities multiply.** The light through the sample is
  $I_\lambda = I_{0,\lambda}\,10^{-A_\lambda}$.

This difference is important again in Part 5.

The sensor measures through a cell with a constant path length. Therefore the
concentration of each dye is its **volume fraction**,
$c_d = V_d / V_\text{total}$.

**What you already have:**

| Name | Type | What it holds |
|---|---|---|
| `DYE_ABSORPTIVITY` | dict | 3 keys: `"R"`, `"Y"`, `"B"`. Each value is a numpy array of 8 floats. |
| `PATHLENGTH_CM` | float | `1.0`. The path length $\ell$. |
| `REFERENCE_SPECTRUM` | numpy array, shape (8,) | The blank reading. This is $I_0$. |
| `CHANNELS` | list of 8 strings | `"ch410"` to `"ch670"`. |

**What you must make:** two functions. Each one returns a numpy array of 8
numbers.

```
predict_absorbance(r_vol, y_vol, b_vol) -> numpy array, shape (8,)
predict_spectrum(r_vol, y_vol, b_vol)   -> numpy array, shape (8,)
```

<details>
<summary>▸ Hints</summary>

- `total` is the sum of the three volumes. There is no other liquid in the
  mixture.
- The sum across the dyes is
  `sum(DYE_ABSORPTIVITY[d] * fractions[d] for d in fractions)`. Numpy makes an
  array of 8 numbers from this expression.
- To go from absorbance to counts, use the Beer-Lambert law in the other
  direction: `I = I0 * 10 ** (-A)`.
- Give `(30, 30, 30)` and then `(90, 90, 90)` to your `predict_absorbance`
  function. Compare the two results. Task 8 examines this again.

</details>

In [ ]:
# ── TASK 2 ──────────────────────────────────────────────────────────────────
def predict_absorbance(r_vol, y_vol, b_vol):
    """Calculate the Beer-Lambert absorbance of the mixture in each channel."""
    total = ___                             # float: the sum of the three volumes
    fractions = {"R": ___, "Y": ___, "B": ___}   # each blank: a float from 0 to 1
    # `dye` takes the values "R", "Y", then "B".
    # DYE_ABSORPTIVITY[dye] is an array of 8 numbers. fractions[dye] is ONE
    # number. numpy multiplies every element of the array by that number.
    # The blank is that product. sum(...) then adds the three arrays together.
    return PATHLENGTH_CM * sum(___ for dye in fractions)


def predict_spectrum(r_vol, y_vol, b_vol):
    """Calculate the expected sensor counts for the mixture."""
    # The blank is an array of 8 numbers: 10.0 to the power of minus the
    # absorbance. Call the function above to get the absorbance.
    return REFERENCE_SPECTRUM * ___


print("A(60, 60, 60) =", np.round(predict_absorbance(60, 60, 60), 3))
print("I(60, 60, 60) =", np.round(predict_spectrum(60, 60, 60), 1))

In [ ]:
# ✅ CHECK — Task 2
c = checker("Task 2 - color prediction")
a_equal = np.asarray(predict_absorbance(60, 60, 60), dtype=float)
c.that(a_equal.shape == (8,), "the absorbance has one value for each channel",
       f"the shape must be (8,). It is {a_equal.shape}.")
c.that(np.allclose(predict_absorbance(1, 0, 0), DYE_ABSORPTIVITY["R"] * PATHLENGTH_CM),
       "pure red gives the absorptivity of the red dye",
       "for a pure red mixture, the absorbance must be eps_R * pathlength")
c.that(np.allclose(a_equal, PATHLENGTH_CM * sum(DYE_ABSORPTIVITY.values()) / 3),
       "equal volumes give the mean of the three absorptivities",
       "the absorbances of the dyes must add together, with the volume "
       "fractions as weights")
c.that(np.allclose(predict_absorbance(30, 30, 30), predict_absorbance(90, 90, 90)),
       "the absorbance changes with the composition, not with the total volume",
       "(30,30,30) and (90,90,90) have the same volume fractions. "
       "Do you divide by the total volume?")
spec = np.asarray(predict_spectrum(60, 60, 60), dtype=float)
c.that(np.all(spec > 0) and np.all(spec < REFERENCE_SPECTRUM),
       "a sample with dye lets less light through than a sample with no dye",
       "each predicted count must be more than 0 and less than REFERENCE_SPECTRUM")
c.that(np.allclose(predict_spectrum(1, 0, 0),
                   REFERENCE_SPECTRUM * 10.0 ** (-np.asarray(predict_absorbance(1, 0, 0)))),
       "the spectrum and the absorbance agree",
       "predict_spectrum must be I0 * 10 ** (-A)")
c.done()

In [ ]:
# The expected color of each experiment in your design. You calculate these
# colors before the robot moves any dye.
design["predicted_hex"] = [spectrum_to_hex(predict_spectrum(r.R, r.Y, r.B))
                           for r in design.itertuples()]
show_swatches(
    [f"#{r.run_order} {r.block[:4]}<br>{r.R:.0f}/{r.Y:.0f}/{r.B:.0f}" for r in design.itertuples()],
    design.predicted_hex,
)

---

## Part 3 · Do the experiments

### The pipeline test (experiment 26 of 26)

Use one experiment to test the full path before you use the other 25. The test
shows that your code can send a job, receive the data, read it, save it, and
show it.

This is the spare experiment in the quota of 25 + 1. It is the most useful
experiment in your budget.

In [ ]:
# Pipeline test. One experiment at the center recipe.
test_spectrum, test_killed = run_experiment(CENTER, CENTER, CENTER, well_index=1)

print("counts         :", test_spectrum)
print("queue stopped? :", test_killed)
print("measured color :", spectrum_to_hex(test_spectrum))
print("expected color :", spectrum_to_hex(predict_spectrum(CENTER, CENTER, CENTER)))
show_swatches(["expected", "measured"],
              [spectrum_to_hex(predict_spectrum(CENTER, CENTER, CENTER)),
               spectrum_to_hex(test_spectrum)])

The measured square is lighter than the expected square. Stray light adds
counts. More counts mean that more light goes through the sample. More light
gives a lower absorbance.

Task 8 examines this error again. There you must give it a name and control it.

### 📋 Task 3 — Do the design

**What you already have:**

| Name | Type | What it holds |
|---|---|---|
| `design` | DataFrame, 25 rows | Your plan. Columns `run_order`, `block`, `R`, `Y`, `B`, `well_index`. |
| `run_experiment(r, y, b, well_index=w)` | function | Returns a **tuple** of 2 items: a numpy array of 8 counts, and a bool. |
| `append_result(record)` | function | Takes one **dict**. Writes one row to the CSV file. |
| `CHANNELS` | list of 8 strings | The channel names, in the same sequence as the counts. |

Write the loop for the campaign. For each row of `design`, in sequence:

1. Call `run_experiment(...)` with the volumes and the well index of that row.
2. Make a record dictionary. It must contain the data of the experiment and the
   8 channel values.
3. Call `append_result(record)`. This writes the record to the disk
   immediately.
4. Stop the loop if the queue stops.

Step 3 is important. Each job takes about 2 minutes. If the queue stops at
experiment 19, and your data is in a Python list only, you lose the full hour.
If the data is on the disk, you keep 18 good measurements.

<details>
<summary>▸ Hints</summary>

- `for row in design.itertuples():` gives you `row.R`, `row.Y`, `row.B`,
  `row.run_order`, `row.well_index`, and `row.block`.
- `run_experiment` returns two values: `(spectrum, killed)`.
- `zip(CHANNELS, spectrum)` puts each channel name with its value. Then
  `record[channel] = value` fills the columns `ch410` to `ch670`.
- Use `break` to stop the loop when `killed` is true. First append the record
  that you already have.

</details>

In [ ]:
# Start a new log file. On the robot, this cell stops if a log file is present.
# It does not write on your data.
if os.path.exists(RESULTS_CSV):
    if USE_HARDWARE:
        raise SystemExit(
            f"The file {RESULTS_CSV} is present. Give it a different name before you "
            "start a new campaign. Or read it, and do not use more of your quota.")
    os.remove(RESULTS_CSV)
print(f"log file: {RESULTS_CSV}")

In [ ]:
# ── TASK 3 ──────────────────────────────────────────────────────────────────
# On the robot, each experiment takes about 2 minutes. Start the loop and wait.
records = []

for row in design.itertuples():
    # run_experiment returns TWO values, so two names are on the left:
    #   spectrum : numpy array of 8 counts
    #   killed   : bool. True if the queue stopped.
    # The first three blanks are floats from `row`. The fourth is an int.
    spectrum, killed = run_experiment(___, ___, ___, well_index=___)

    record = {
        "run_order": int(row.run_order),
        "block": row.block,
        "R": float(row.R), "Y": float(row.Y), "B": float(row.B),
        "well_index": int(row.well_index),
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    # zip(CHANNELS, <8 numbers>) pairs each name with one number. In the loop,
    # `channel` is a string such as "ch410", and `value` is one count.
    # First blank : the array of 8 counts.
    # Second blank: the number for this channel.
    for channel, value in zip(CHANNELS, ___):
        record[channel] = ___

    append_result(record)          # write to the disk first, every time
    records.append(record)
    print(f"experiment {row.run_order:2d}/{len(design)}  "
          f"R={row.R:5.1f} Y={row.Y:5.1f} B={row.B:5.1f}  ->  {spectrum_to_hex(spectrum)}")

    if ___:      # the bool that run_experiment gave you
        print("The queue stopped. All the data up to here is on the disk.")
        break

results = pd.DataFrame(records)
print(f"\n{len(results)} experiments are complete")

In [ ]:
# ✅ CHECK — Task 3
c = checker("Task 3 - data collection")
c.that(len(results) == len(design), f"all {len(design)} experiments are complete",
       f"{len(results)} of {len(design)} experiments are complete")
c.that(all(ch in results.columns for ch in CHANNELS),
       "all 8 sensor channels have a column",
       "a channel column is not present. Put the spectrum into the record.")
c.that(results[CHANNELS].notna().all().all(), "no sensor value is missing",
       "one or more channel values are NaN")
c.that((results[CHANNELS] > 0).all().all(), "each reading is more than zero",
       "one or more channels read zero or less")
on_disk = load_results()
c.that(len(on_disk) == len(results), f"{len(results)} rows are in {RESULTS_CSV}",
       f"the CSV file has {len(on_disk)} rows, but you have {len(results)} "
       "experiments. Is append_result inside the loop?")
c.that(results.run_order.is_monotonic_increasing,
       "the experiments ran in the random sequence",
       "run_order does not increase")
c.done()

---

## Part 4 · Change the spectra into a response

### 📋 Task 4 — Make the response variables

Each experiment gives you 8 numbers. ANOVA needs one number for each
experiment. The selection of that number is part of the model. It is not a
formality.

Counts are a bad selection. The Beer-Lambert law shows that the counts decrease
exponentially with the quantity of dye. Absorbance is the correct scale:

$$A_\lambda = -\log_{10}\!\left(\frac{I_\lambda}{I_{0,\lambda}}\right)$$

On the absorbance scale, the dyes **add together**. On the count scale, they
**multiply**. A model on the correct scale needs fewer interaction terms. This
is a general rule, and it is more useful than this experiment: before you add
interaction terms, look at the scale of your response.

**What you already have:**

| Name | Type | What it holds |
|---|---|---|
| `results` | DataFrame, 25 rows | `run_order`, `block`, `R`, `Y`, `B`, `well_index`, `timestamp`, and `ch410` to `ch670`. |
| `REFERENCE_SPECTRUM` | numpy array, shape (8,) | The blank reading, $I_0$. |
| `predict_spectrum(r, y, b)` | function | The one that you wrote in Task 2. |
| `CHANNELS` | list of 8 strings | The channel names. |

Calculate three quantities:

- `A_ch410` to `A_ch670` — the absorbance in each channel.
- `A_total` — the sum of the 8 channels. This is your **primary response**. It
  is one number for the quantity of light that the mixture absorbs.
- `rmse_pred` — the RMSE between the measured spectrum and your prediction from
  Task 2. The Bayesian optimizer in [`BO.ipynb`](./BO.ipynb) makes this same
  quantity as small as possible. Here you use it to examine a model.

<details>
<summary>▸ Hints</summary>

- Absorbance from counts: `-np.log10(spectrum / reference)`.
- `np.clip(spectrum, 1.0, None)` before the division prevents `inf` from a
  reading of zero.
- `results[CHANNELS].to_numpy()` gives you an array of 25 rows and 8 columns.
- RMSE between two arrays of 8 numbers:
  `np.sqrt(np.mean((a - b) ** 2))`.
- In the last list, `row` is a namedtuple. To get the 8 measured counts from it,
  use `np.array([getattr(row, ch) for ch in CHANNELS])`.

</details>

In [ ]:
# ── TASK 4 ──────────────────────────────────────────────────────────────────
def absorbance(spectrum, reference=REFERENCE_SPECTRUM):
    """Calculate the Beer-Lambert absorbance in each channel, from the counts."""
    spectrum = np.clip(np.asarray(spectrum, dtype=float), 1.0, None)
    # The blank is a numpy array of 8 numbers. Divide the counts by the
    # reference, then take -log10 of the result.
    return ___


# The absorbance in each channel
A = np.array([absorbance(row) for row in results[CHANNELS].to_numpy()])
for i, channel in enumerate(CHANNELS):
    results["A_" + channel] = A[:, i]

# The primary response: the total absorbance across the spectrum.
# A is a numpy array with shape (25, 8): 25 experiments, 8 channels. You need
# ONE number for each experiment, so add ACROSS the 8 channels. Pattern 7 in
# Part 1b shows that axis=1 adds across each row.
results["A_total"] = ___

# The distance from each measurement to the prediction in Task 2.
# The blank is ONE float for each experiment. Inside the list, `row` is one row
# of results, so row.R, row.Y, and row.B are the volumes. To collect the 8
# measured counts from `row`, use:
#     np.array([getattr(row, ch) for ch in CHANNELS])
# Then take the RMSE against predict_spectrum(row.R, row.Y, row.B).
results["rmse_pred"] = [
    ___
    for row in results.itertuples()
]

results[["run_order", "block", "R", "Y", "B", "A_total", "rmse_pred"]]

In [ ]:
# ✅ CHECK — Task 4
c = checker("Task 4 - response variables")
first = results[CHANNELS].to_numpy()[0]
c.that(np.allclose(absorbance(first), -np.log10(first / REFERENCE_SPECTRUM), atol=1e-9),
       "the absorbance agrees with -log10(I / I0)",
       "the formula must be -log10(spectrum / reference)")
c.that(np.allclose(absorbance(REFERENCE_SPECTRUM), 0.0),
       "a reading with no dye gives an absorbance of zero",
       "the absorbance of REFERENCE_SPECTRUM must be 0 in each channel")
c.that(all("A_" + ch in results.columns for ch in CHANNELS),
       "each channel has an absorbance column",
       "one or more A_ch* columns are not present")
c.that(np.allclose(results.A_total, results[["A_" + ch for ch in CHANNELS]].sum(axis=1)),
       "A_total is the sum of the 8 channels",
       "A_total must be the sum of the absorbance in each channel")
c.that((results.A_total > 0).all(), "each mixture absorbs some light",
       "one or more A_total values are 0 or less. This means that the sample "
       "is brighter than the blank.")
c.that((results.rmse_pred >= 0).all() and results.rmse_pred.notna().all(),
       "rmse_pred has a value for each experiment",
       "rmse_pred has negative values or missing values")
c.done()

---

## Part 5 · Calculate the effects

### 📋 Task 5 — Calculate the effects manually

Calculate the effects before you use a software library. A 2³ factorial design
is small, and the arithmetic shows you how the method operates.

**What you already have:**

| Name | Type | What it holds |
|---|---|---|
| `results` | DataFrame, 25 rows | Now also has `A_ch410` to `A_ch670`, `A_total`, and `rmse_pred`. |
| `LOW`, `HIGH`, `CENTER` | floats | The levels that you selected in Task 1. |
| `itertools` | module | `itertools.combinations(FACTORS, 2)` gives the 3 pairs. |

**Step 1. Code the factors.** Change each volume to a scale where the low level
is −1 and the high level is +1:

$$x_R = \frac{V_R - \text{CENTER}}{(\text{HIGH} - \text{LOW})/2}$$

**Step 2. Calculate the main effects.** The effect of a factor is the mean
response at the high level, minus the mean response at the low level:

$$\text{Effect}_R = \bar{y}_{x_R=+1} - \bar{y}_{x_R=-1}$$

The design is balanced. Therefore all 16 experiments give data about each
effect. OFAT cannot do this.

**Step 3. Calculate the interactions.** An interaction column is the product of
two coded columns. Multiply the columns. Then calculate the same difference of
the means. For $N$ factorial experiments, each effect has this equation:

$$\text{Effect} = \frac{2}{N}\sum_i x_i\, y_i$$

$x_i$ is the value in the applicable column for experiment $i$. The `RB`
interaction has this meaning: the effect of red changes when blue moves from the
low level to the high level. The interaction is one half of that change.

<details>
<summary>▸ Hints</summary>

- The half range is `(HIGH - LOW) / 2`. Therefore the coded value is
  `(volume - CENTER) / half_range`.
- `factorial["xR"] * factorial["xB"]` makes the `RB` column.
- `N` is `len(factorial)`. This is 16 if all your replicates are complete.
  Therefore the divisor is `len(factorial) / 2`.
- The two equations in the brief give the same number. Use the difference of
  the means for the main effects, and the sum equation for the interactions.
  The check cell compares them.

</details>

In [ ]:
# ── TASK 5 ──────────────────────────────────────────────────────────────────
RESPONSE = "A_total"
FACTORS = ("R", "Y", "B")

# Keep the 16 factorial rows only. .copy() makes a separate table, so that new
# columns do not change `results`.
factorial = results[results.block == "factorial"].copy()
half_range = ___     # float: half of the distance from LOW to HIGH

# Step 1: code the factors to -1 and +1.
# `f` takes the values "R", "Y", then "B". The blank is the column of volumes
# for that factor, as a Series of 16 numbers.
for f in FACTORS:
    factorial["x" + f] = (___ - CENTER) / half_range

y = factorial[RESPONSE]
n_runs = len(factorial)
effects = {}

# Step 2: the main effects, as a difference of the means
for f in FACTORS:
    high_mean = y[factorial["x" + f] > 0].mean()
    low_mean = y[factorial["x" + f] < 0].mean()
    effects[f] = ___     # float: the high mean minus the low mean

# Step 3: the interactions, from the product columns
# `a` and `b` are two different factor names, such as "R" and "B".
# First blank : a Series of 16 numbers. Multiply the two coded columns
#               together, element by element.
# Second blank: a float. The equation in the brief divides by N / 2.
for a, b in itertools.combinations(FACTORS, 2):
    factorial[f"x{a}{b}"] = ___
    effects[a + b] = (factorial[f"x{a}{b}"] * y).sum() / ___

# The three-way column: multiply all three coded columns together.
factorial["xRYB"] = ___
effects["RYB"] = (factorial["xRYB"] * y).sum() / ___

for name, value in effects.items():
    print(f"  {name:>4s}  {value:+.4f}")

In [ ]:
# ✅ CHECK — Task 5
c = checker("Task 5 - effect estimates")
c.that(set(factorial[["xR", "xY", "xB"]].to_numpy().ravel()) == {-1.0, 1.0},
       "the factors are coded to -1 and +1",
       "a coded column has a value that is not -1 and not +1. Check the half range.")
c.that(all(abs(factorial["x" + f].sum()) < 1e-9 for f in FACTORS),
       "the design is balanced. Each coded column adds up to zero.",
       "a coded column does not add up to zero. The design is not balanced.")
for f in FACTORS:
    contrast = (factorial["x" + f] * factorial[RESPONSE]).sum() / (len(factorial) / 2)
    c.that(abs(contrast - effects[f]) < 1e-9,
           f"the main effect {f} agrees with the sum equation",
           f"main effect {f}: the difference of the means gives {effects[f]:.4f}, "
           f"and the sum equation gives {contrast:.4f}")
c.that(np.allclose(factorial.xRB, factorial.xR * factorial.xB),
       "each interaction column is a product of two coded columns",
       "xRB must be xR multiplied by xB")
c.that(len(effects) == 7,
       "7 effects: 3 main, 3 two-way, and 1 three-way",
       f"there must be 7 effects. There are {len(effects)}.")
c.done()

In [ ]:
# A Pareto chart of the effects. It shows which terms are large.
import matplotlib.pyplot as plt

order = sorted(effects, key=lambda k: abs(effects[k]))
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.barh(order, [abs(effects[k]) for k in order],
        color=["#4c72b0" if len(k) == 1 else "#dd8452" for k in order])
ax.set_xlabel(f"absolute effect on {RESPONSE}")
ax.set_title("Pareto chart of the effects (blue = main, orange = interaction)")
fig.tight_layout()
plt.show()

In [ ]:
# Main effect plots. Each plot shows the mean response at each level.
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharey=True)
for ax, f in zip(axes, FACTORS):
    means = factorial.groupby(f)[RESPONSE].mean()
    ax.plot(means.index, means.values, "o-", color="#4c72b0")
    ax.axhline(factorial[RESPONSE].mean(), ls=":", c="grey")
    ax.set_xlabel(f"{f} volume (uL)")
    ax.set_title(f"{f}: effect = {effects[f]:+.3f}")
axes[0].set_ylabel(RESPONSE)
fig.suptitle("Main effects. A steeper line shows a larger effect.")
fig.tight_layout()
plt.show()

In [ ]:
# Interaction plots. Parallel lines show no interaction. Lines that cross, or
# lines that move apart, show that the effect of one factor changes with the
# level of the other factor.
pairs = list(itertools.combinations(FACTORS, 2))
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharey=True)
for ax, (a, b) in zip(axes, pairs):
    for level, sub in factorial.groupby(b):
        means = sub.groupby(a)[RESPONSE].mean()
        ax.plot(means.index, means.values, "o-", label=f"{b} = {level:.0f}")
    ax.set_xlabel(f"{a} volume (uL)")
    ax.set_title(f"{a}x{b}: effect = {effects[a + b]:+.3f}")
    ax.legend(fontsize=8)
axes[0].set_ylabel(RESPONSE)
fig.suptitle("Interaction plots")
fig.tight_layout()
plt.show()

### 📋 Task 6 — ANOVA

The size of an effect is not enough. An effect of 0.07 is important if the noise
is 0.01. The same effect is not important if the noise is 0.5.

**ANOVA** divides the total variation in the response into one part for each
model term, and one residual part. Then it examines each part and asks this
question: is this part larger than noise alone?

Your design has replicates. Therefore the residual is a true measurement of
**pure error**. Pure error is the difference between two experiments with the
same recipe.

**What you already have:**

| Name | Type | What it holds |
|---|---|---|
| `factorial` | DataFrame, 16 rows | Now also has the coded columns `xR`, `xY`, `xB`, `xRY`, `xRB`, `xYB`, `xRYB`. |
| `effects` | dict | 7 keys: `"R"`, `"Y"`, `"B"`, `"RY"`, `"RB"`, `"YB"`, `"RYB"`. Each value is a float. |
| `RESPONSE` | str | `"A_total"`. The name of the response column. |

Fit the full model with three factors, on the coded columns. Then make the ANOVA
table. Use the coded columns, and not the volumes. With `xR`, `xY`, and `xB` in
±1 units, each coefficient is one half of the effect that you calculated
manually. The check cell tests this.

In the formula, `a * b` means "a, b, and the interaction of a and b". Therefore
one term gives you the full factorial model.

<details>
<summary>▸ Hints</summary>

- The formula is `"A_total ~ xR * xY * xB"`. You can also make the string from
  the `RESPONSE` variable. Then you can change the response with one edit.
- `smf.ols(formula, data=factorial).fit()` fits the model.
- `sm.stats.anova_lm(model, typ=2)` makes the table. Type II is the usual
  selection for a balanced design. For a balanced factorial design, types I, II,
  and III give the same result.

</details>

In [ ]:
# ── TASK 6 ──────────────────────────────────────────────────────────────────
import statsmodels.api as sm
import statsmodels.formula.api as smf

formula = ___                                 # str, in the form "response ~ terms"
model = smf.ols(___, data=factorial).fit()    # the formula string from above
anova_table = sm.stats.anova_lm(model, typ=___)   # int: the ANOVA type

print(f"residual standard deviation: {np.sqrt(model.mse_resid):.4f}")
print(f"R-squared: {model.rsquared:.4f}\n")
anova_table.round(5)

In [ ]:
# ✅ CHECK — Task 6
c = checker("Task 6 - ANOVA")
term_map = {"xR": "R", "xY": "Y", "xB": "B", "xR:xY": "RY",
            "xR:xB": "RB", "xY:xB": "YB", "xR:xY:xB": "RYB"}
c.that(set(term_map) <= set(model.params.index),
       "the model has all 7 factorial terms",
       "a term is not present in the model. Use the formula 'xR * xY * xB'.")
if set(term_map) <= set(model.params.index):
    worst = max(abs(2 * model.params[t] - effects[n]) for t, n in term_map.items())
    c.that(worst < 1e-8,
           "each coefficient is one half of your manual effect",
           f"the largest difference between 2*coefficient and your effect "
           f"is {worst:.2e}")
c.that(model.df_resid == len(factorial) - 8,
       f"{int(len(factorial) - 8)} residual degrees of freedom, from the replicates",
       "the residual degrees of freedom are incorrect. Without replicates, "
       "ANOVA cannot estimate the error.")
c.that("PR(>F)" in anova_table.columns, "the ANOVA table has p-values",
       "the p-value column is not present. Check the anova_lm call.")
c.done()

significant = anova_table[anova_table["PR(>F)"] < 0.05].index.tolist()
print("\nsignificant at alpha = 0.05:", ", ".join(t for t in significant if t != "Residual"))

In [ ]:
# A test for curvature. A design with two levels fits flat surfaces. It cannot
# find a curve. The center points can find a curve. If the response were linear,
# the mean of the center experiments would be equal to the mean of the corner
# experiments.
centers = results[results.block == "center"]
corner_mean, center_mean = factorial[RESPONSE].mean(), centers[RESPONSE].mean()
pooled_sd = np.sqrt(model.mse_resid)
se = pooled_sd * np.sqrt(1 / len(factorial) + 1 / len(centers))
t_stat = (center_mean - corner_mean) / se

from scipy import stats
p_curv = 2 * (1 - stats.t.cdf(abs(t_stat), df=model.df_resid))
print(f"corner mean {corner_mean:.4f}   center mean {center_mean:.4f}")
print(f"curvature   {center_mean - corner_mean:+.4f}   t = {t_stat:+.2f}   p = {p_curv:.3f}")
print("\n" + ("The curvature is significant. A design with two levels is not "
              "sufficient. You must add axial points and make a central "
              "composite design."
              if p_curv < 0.05 else
              "The curvature is not significant. Across this range, a model with "
              "linear terms and interaction terms is sufficient."))

In [ ]:
# The scale of the response is important. Fit the model again on the COUNTS in
# one channel, in place of the absorbance. Then look at the interaction terms.
factorial_raw = factorial.copy()
factorial_raw["raw"] = factorial_raw["ch550"]
raw_model = smf.ols("raw ~ xR * xY * xB", data=factorial_raw).fit()
log_model = smf.ols("A_ch550 ~ xR * xY * xB", data=factorial).fit()

comparison = pd.DataFrame({
    "p (counts)": sm.stats.anova_lm(raw_model, typ=2)["PR(>F)"],
    "p (absorbance)": sm.stats.anova_lm(log_model, typ=2)["PR(>F)"],
}).round(4)
print("Channel ch550: the same data on two response scales\n")
print(comparison)
print("\nThe Beer-Lambert law makes the counts multiply and the absorbances add.")
print("Therefore an interaction on the count scale can come from the scale, and")
print("not from the chemistry. The selection of the response is part of the")
print("design.")

---

## Part 6 · Plot the design and the response

### 📋 Task 7 — Ternary diagrams

A **ternary diagram** shows three components as proportions. Each corner is 100%
of one dye. Each point in the triangle is a mixture. This is the correct diagram
for a formulation with three dyes, and the assignment needs two of them.

A ternary diagram shows the composition only. Two experiments with the same
ratios are at the same point, also when the total volumes are different. For
example, `(30, 30, 30)` and `(90, 90, 90)` are one point. If the total volume
changes your response, this diagram does not show it. Task 8 tests the total
volume.

**What you already have:**

| Name | Type | What it holds |
|---|---|---|
| `results` | DataFrame, 25 rows | All the data, with the response columns from Task 4. |
| `RESPONSE` | str | `"A_total"`. |
| `px` | module | `plotly.express`, imported in the cell below. |

Change the volumes into fractions. Then make two figures:

1. **Sample positions** — each experiment in the design, with a color for each
   `block`.
2. **Response surface** — each experiment, with a color for `A_total`.

<details>
<summary>▸ Hints</summary>

- The fraction of red is `results.R / (results.R + results.Y + results.B)`.
- `px.scatter_ternary(df, a="fR", b="fY", c="fB", color="block")`.
- For the response surface, use `color=RESPONSE` and
  `color_continuous_scale="Viridis"`.
- `size=...` and `hover_data=["run_order", "R", "Y", "B"]` let you examine each
  point.

</details>

In [ ]:
# ── TASK 7 ──────────────────────────────────────────────────────────────────
import plotly.express as px

# Each blank here is a Series of 25 numbers, one for each experiment.
# pandas adds and divides row by row, so results.R / total_volume gives the
# fraction of red in every experiment at the same time.
total_volume = ___          # Series: the three volumes added together
results["fR"] = ___         # Series: the red volume divided by the total
results["fY"] = ___         # Series: the yellow volume divided by the total
results["fB"] = ___         # Series: the blue volume divided by the total
results["total_volume"] = total_volume

# Figure 1 — the sample positions
# The a, b, and c blanks are STRINGS: the names of the three fraction columns.
# The color blank is also a string: the name of the column that sets the color.
fig1 = px.scatter_ternary(
    results, a=___, b=___, c=___,
    color=___,
    hover_data=["run_order", "R", "Y", "B", "total_volume"],
    title="Figure 1 - DoE sample positions",
)
fig1.update_traces(marker=dict(size=12, line=dict(width=1, color="white")))
fig1.show()

# Figure 2 — the response surface
fig2 = px.scatter_ternary(
    results, a="fR", b="fY", c="fB",
    color=___,      # str: the name of the response column
    color_continuous_scale="Viridis",
    hover_data=["run_order", "R", "Y", "B", "total_volume"],
    title=f"Figure 2 - response surface ({RESPONSE})",
)
fig2.update_traces(marker=dict(size=14, line=dict(width=1, color="white")))
fig2.show()

In [ ]:
# ✅ CHECK — Task 7
c = checker("Task 7 - ternary diagrams")
c.that(all(col in results.columns for col in ("fR", "fY", "fB")),
       "the composition fractions are present",
       "one or more of the fR, fY, and fB columns are not present")
c.that(np.allclose(results.fR + results.fY + results.fB, 1.0),
       "the fractions add up to 1 in each experiment",
       "the fractions do not add up to 1. Divide each volume by the total "
       "volume of that experiment.")
c.that(len(results.groupby(["fR", "fY", "fB"])) < len(results),
       "some experiments have the same composition, because of the replicates",
       "each experiment has a different composition. Are your replicates "
       "correct?")
c.done()

# The measured colors, in the sequence of the experiments.
show_swatches([f"#{r.run_order}<br>{r.block[:4]}" for r in results.itertuples()],
              [spectrum_to_hex(np.array([getattr(r, ch) for ch in CHANNELS]))
               for r in results.itertuples()])

---

## Part 7 · Find the errors

### 📋 Task 8 — Examine the residuals

The residuals are the part of the data that the model cannot explain. If your
model is correct, and only random noise stays, the residuals have no pattern.
They have a normal distribution, and they are independent of all the variables
that are not in the model.

A pattern in the residuals shows a problem. Make three plots, and answer three
questions:

| Plot | Question | Sign of a problem |
|---|---|---|
| **Normal probability plot** | Do the residuals have a normal distribution? | The points move away from the straight line |
| **Residual against run order** | Does something drift during the hour? | A trend, or a path that moves |
| **Residual against plate column** | Does the well position change the result? | A slope, or a step |

The third plot examines the stray light problem from Part 0. Randomization made
the well position independent of your factors. Therefore a position effect
cannot change an effect estimate. It can only make the residuals larger. The
residuals are the correct place to find it.

**What you already have:**

| Name | Type | What it holds |
|---|---|---|
| `model` | statsmodels result | `model.fittedvalues` and `model.resid` are each a Series of 16 numbers. |
| `factorial` | DataFrame, 16 rows | Has `run_order`, `well_index`, the coded columns, and `A_total`. |
| `plate_position(w)` | function | Takes an int. Returns a **tuple** `(row_letter, column_number)`. |
| `plt` | module | `matplotlib.pyplot`. |

`plate_position` returns a tuple, so `plate_position(5)[0]` is the letter and
`plate_position(5)[1]` is the column number.

<details>
<summary>▸ Hints</summary>

- `model.fittedvalues` and `model.resid` have the same index as `factorial`.
- `scipy.stats.probplot(residuals, dist="norm", plot=ax)` makes the normal
  probability plot and its line.
- `plate_position(w)[1]` is the column number of well `w`.
- To get a number in place of a visual test, use
  `scipy.stats.pearsonr(factorial.plate_col, factorial.residual)`.

</details>

In [ ]:
# ── TASK 8 ──────────────────────────────────────────────────────────────────
from scipy import stats

factorial["fitted"] = ___        # Series of 16: the values that the model predicts
factorial["residual"] = ___      # Series of 16: the response minus the fitted value
# In the list below, `w` is one well index (an int). The blank must give the
# COLUMN NUMBER for that well, so take item [1] of the tuple.
factorial["plate_col"] = [___ for w in factorial.well_index]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

# (a) Figure 4 — the normal probability plot of the residuals
# The blank is the Series of 16 residuals.
stats.probplot(___, dist="norm", plot=axes[0])
axes[0].set_title("Figure 4 - normal probability plot")

# (b) residual against run order — this finds a drift during the session
# The blank is a column name: the sequence number of each experiment.
axes[1].scatter(factorial.___, factorial.residual, color="#4c72b0")
axes[1].axhline(0, ls=":", c="grey")
axes[1].set_xlabel("run order")
axes[1].set_ylabel("residual")
axes[1].set_title("Residuals against run order")

# (c) residual against plate column — this finds a position effect
# The blank is a column name: the plate column that you made above.
axes[2].scatter(factorial.___, factorial.residual, color="#dd8452")
axes[2].axhline(0, ls=":", c="grey")
axes[2].set_xlabel("plate column")
axes[2].set_title("Residuals against plate column")

fig.tight_layout()
plt.show()

r_pos, p_pos = stats.pearsonr(factorial.plate_col, factorial.residual)
r_time, p_time = stats.pearsonr(factorial.run_order, factorial.residual)
print(f"residual against plate column : r = {r_pos:+.3f}  p = {p_pos:.4f}")
print(f"residual against run order    : r = {r_time:+.3f}  p = {p_time:.4f}")

In [ ]:
# ✅ CHECK — Task 8
c = checker("Task 8 - residual diagnostics")
c.that("residual" in factorial.columns and "fitted" in factorial.columns,
       "the fitted values and the residuals are present",
       "the fitted column or the residual column is not present")
c.that(abs(factorial.residual.sum()) < 1e-8,
       "the residuals add up to zero, as least squares needs",
       "the residuals do not add up to zero. Are these the residuals of the model?")
c.that(np.allclose(factorial.fitted + factorial.residual, factorial[RESPONSE]),
       "fitted plus residual gives the response again",
       "the fitted values and the residuals do not agree with the response")
c.that(factorial.plate_col.between(1, 12).all(),
       "the plate columns are in the range 1 to 12",
       "plate_col is outside the range. plate_position returns "
       "(row_letter, column_number).")
c.done()

In [ ]:
# Does the total volume change the response? Your design has a test for this.
# The corners (LOW, LOW, LOW) and (HIGH, HIGH, HIGH) have the same dye ratios,
# but their total volumes are different by a factor of three. The center points
# have the same ratios again.
same_ratio = results[np.isclose(results.fR, 1 / 3) & np.isclose(results.fY, 1 / 3)]
print("Experiments with equal dye fractions and different total volumes:\n")
print(same_ratio[["run_order", "block", "R", "Y", "B", "total_volume", RESPONSE]]
      .sort_values("total_volume").to_string(index=False))

if len(same_ratio) > 2:
    r_vol, p_vol = stats.pearsonr(same_ratio.total_volume, same_ratio[RESPONSE])
    print(f"\n{RESPONSE} against total volume, at one composition: "
          f"r = {r_vol:+.3f}  p = {p_vol:.3f}")
    print("\nA large p-value shows that the sensor responds to the concentration")
    print("only. The total volume is then not a hidden variable. A small p-value")
    print("shows that the path length changes with the quantity of liquid. Then")
    print("you must put the total volume in your model.")

### 📋 Task 9 — Expected color against measured color

This is the primary question in the assignment. You predict the color for a set
of RYB volumes. Then you measure the color and compare the two.

Your five validation experiments are not in the model. Therefore they give an
independent test.

**What you already have:**

| Name | Type | What it holds |
|---|---|---|
| `results` | DataFrame, 25 rows | The `block` column holds `"factorial"`, `"center"`, or `"validation"`. |
| `predict_spectrum(r, y, b)` | function | Your function from Task 2. Returns 8 numbers. |
| `factorial` | DataFrame, 16 rows | The data that the model used. |
| `CHANNELS` | list of 8 strings | The channel names. |

Make two items:

1. **Figure 3, a parity plot.** Put the expected counts on one axis and the
   measured counts on the other axis. Plot one point for each channel of each
   validation experiment. This gives 40 points. Add the line $y = x$. A point on
   the line shows a correct prediction. Points above the line show that the
   equipment measures more light than the model predicts. Stray light that adds
   to the signal causes this.
2. **A comparison of the colors.** Show the expected color and the measured
   color of each validation experiment. A parity plot does not tell you if a
   person can see the difference.

<details>
<summary>▸ Hints</summary>

- `results[results.block == "validation"]` selects the validation experiments.
- Put the predictions together with
  `np.array([predict_spectrum(r.R, r.Y, r.B) for r in validation.itertuples()])`.
- `.ravel()` makes each array of 5 rows and 8 columns into one long array. Then
  you can plot one against the other.
- For the diagonal line: `lims = [min(...), max(...)]; ax.plot(lims, lims, "k--")`.
- `show_swatches(labels, hexes)` takes two lists of the same length. Put the
  expected color and the measured color one after the other.

</details>

In [ ]:
# ── TASK 9 ──────────────────────────────────────────────────────────────────
validation = results[results.block == ___].copy()   # str: the block name

# The blank calls your Task 2 function with the three volumes of `row`.
# np.array(...) then stacks the 5 results into one array with shape (5, 8).
predicted = np.array([___ for row in validation.itertuples()])
measured = validation[CHANNELS].to_numpy()          # shape (5, 8)

# Figure 3 — the parity plot
fig, ax = plt.subplots(figsize=(4.8, 4.8))
# scatter() needs two flat lists of the same length. Both arrays have shape
# (5, 8). Use .ravel() to change each one into 40 numbers in a row.
ax.scatter(___, ___, alpha=0.75, color="#4c72b0")
lims = [min(predicted.min(), measured.min()) * 0.95,
        max(predicted.max(), measured.max()) * 1.05]
ax.plot(lims, lims, "k--", lw=1, label="correct prediction")
ax.set_xlabel("expected counts")
ax.set_ylabel("measured counts")
ax.set_title("Figure 3 - expected against measured")
ax.legend()
fig.tight_layout()
plt.show()

bias = float(np.mean(measured - predicted))
print(f"mean signed error (measured - expected): {bias:+.1f} counts")
print(f"RMSE: {np.sqrt(np.mean((measured - predicted) ** 2)):.1f} counts")

# The color comparison
labels, hexes = [], []
for row, pred_spec in zip(validation.itertuples(), predicted):
    labels += [f"#{row.run_order} expect", f"#{row.run_order} measur"]
    hexes += [spectrum_to_hex(pred_spec),
              spectrum_to_hex(np.array([getattr(row, ch) for ch in CHANNELS]))]
show_swatches(labels, hexes)

In [ ]:
# ✅ CHECK — Task 9
c = checker("Task 9 - prediction accuracy")
c.that(len(validation) == 5, "5 validation experiments",
       f"there must be 5 validation experiments. There are {len(validation)}.")
c.that(predicted.shape == measured.shape == (len(validation), 8),
       "the expected array and the measured array have the shape (n, 8)",
       f"the shapes are {predicted.shape} and {measured.shape}")
c.that(not validation.index.isin(factorial.index).any(),
       "the model does not contain the validation experiments",
       "a validation experiment is also in the training data")
c.that(abs(bias) > 0, "there is a prediction error to examine",
       "the prediction error is exactly zero. This is not usual.")
c.done()
print(f"\nThe sign of the error is the clue. It is "
      f"{'positive' if bias > 0 else 'negative'}. Therefore the equipment "
      f"measures\n{'MORE' if bias > 0 else 'LESS'} light than the Beer-Lambert "
      "model predicts.\nWhich mechanism in Part 0 gives this result?")

### 📋 Task 10 (optional) — Calibrate the dyes with your own data

The predictions above use the `DYE_ABSORPTIVITY` values from Part 1. On the real
equipment, these values are approximate. Your dye containers have their own
concentrations, and your sensor has its own response. You can calculate better
values from the experiments that you already did.

The absorbance is linear in the volume fractions. For $n$ experiments:

$$\underbrace{\mathbf{A}}_{n \times 8} = \underbrace{\mathbf{C}}_{n \times 3}\;\underbrace{\mathbf{E}}_{3 \times 8}$$

$\mathbf{C}$ contains the volume fractions. $\mathbf{E}$ contains the
absorptivity spectrum of each dye. This is a least squares problem, and
`np.linalg.lstsq` solves it in one line. Fit the model on the factorial
experiments. Then examine the validation predictions again.

**What you already have:**

| Name | Type | What it holds |
|---|---|---|
| `C` | numpy array, shape (16, 3) | The volume fractions of the 16 factorial experiments. |
| `A_obs` | numpy array, shape (16, 8) | The measured absorbances of those experiments. |
| `C_val` | numpy array, shape (5, 3) | The volume fractions of the 5 validation experiments. |
| `predicted`, `measured` | numpy arrays, shape (5, 8) | From Task 9. |

<details>
<summary>▸ Hints</summary>

- `C = train[["fR", "fY", "fB"]].to_numpy()` and
  `A_obs = train[["A_" + ch for ch in CHANNELS]].to_numpy()`.
- `E, *_ = np.linalg.lstsq(C, A_obs, rcond=None)` gives an array of 3 rows and
  8 columns.
- Make the predictions again with the new `E`, and compare the RMSE with the
  value from Task 9.
- A small improvement is also a result. It shows that the model is correct, and
  that the error is systematic and not a calibration problem.

</details>

In [ ]:
# ── TASK 10 (optional) ──────────────────────────────────────────────────────
train = results[results.block == "factorial"]
C = train[["fR", "fY", "fB"]].to_numpy()
A_obs = train[["A_" + ch for ch in CHANNELS]].to_numpy()

# lstsq(X, Y) finds the array E that makes X @ E as near to Y as possible.
# First blank: the fractions, shape (16, 3). Second blank: the absorbances,
# shape (16, 8). E_fitted then has shape (3, 8).
E_fitted, *_ = np.linalg.lstsq(___, ___, rcond=None)

print("fitted absorptivities (rows: R, Y, B. columns: the 8 channels)")
print(np.round(E_fitted, 3))

# Predict the validation experiments again, with the new absorptivities
C_val = validation[["fR", "fY", "fB"]].to_numpy()
# The blank is the matrix product of C_val (5, 3) and E_fitted (3, 8). The
# result has shape (5, 8). The @ symbol does a matrix product in numpy.
predicted_fitted = REFERENCE_SPECTRUM * 10.0 ** (-(___))

rmse_before = np.sqrt(np.mean((measured - predicted) ** 2))
rmse_after = np.sqrt(np.mean((measured - predicted_fitted) ** 2))
print(f"\nvalidation RMSE, given absorptivities : {rmse_before:8.1f} counts")
print(f"validation RMSE, fitted absorptivities: {rmse_after:8.1f} counts")

---

## Part 8 · Write the report

The code is complete. The assignment is not complete. This part is a technical
writing task.

Write your answers in the markdown cells below. Use prose. Give the numbers from
your own results.

### 8.1 — Main effects and interactions

Use your ANOVA table and your effect estimates.

- List the factors that have a significant effect. Give the effect size and the
  p-value of each one.
- List the significant interactions. Give the physical meaning of each one. An
  `R×B` interaction shows that the effect of red changes when blue changes.
  Describe that change.
- List the terms that are not significant. Tell what you get if you remove them
  from the model.
- Tell if the curvature is significant. Tell what your answer shows about a
  design with two levels.

*Your answer:*

> …

### 8.2 — Random error and systematic error

Give a name to each error that you found. Give the data that shows it.

- **Random error.** What is your pure error standard deviation? Where does it
  come from? Which plot shows that this error is random?
- **Systematic error.** What do you see in the plot of the residuals against the
  plate column? What do you see in the mean signed error of Task 9? What is the
  physical mechanism?
- Explain why replication decreases the effect of the first error, but not the
  effect of the second error.
- Was the normal probability plot a straight line? If not, tell what the shape
  shows.

*Your answer:*

> …

### 8.3 — How to decrease these errors

The assignment asks how you can decrease the effect of these errors. Answer from
two directions. A good experimentalist has methods at the design stage and other
methods at the analysis stage.

**At the design stage**, think about these methods:

- Randomization. What did it give you already?
- Blocks on the plate row, or on the plate column.
- More replicates.
- More center points.
- A blank measurement between the samples.
- A constant total volume.
- A control experiment with a cover against the light.

**At the analysis stage**, think about these methods:

- The well position as a covariate, or as a block term in the model.
- Subtraction of a measured stray light offset.
- A different scale for the response.
- Weights from the precision of each measurement.
- Robust regression.

For each method, tell which error it controls and what it costs. The cost can be
experiments, time, or degrees of freedom. A method with no cost is usually a
method that you did not examine fully.

*Your answer:*

> …

### 8.4 — Summary

Write three to five sentences.

- What did you learn about this system?
- What will you do differently with the next 26 experiments?
- Your data made a new question that your data cannot answer. What is it?

*Your answer:*

> …

---

## Part 9 · Checklist before you submit

This table shows each item in the assignment, and its position in this notebook.

| Item | Position | Done |
|---|---|---|
| A DoE with good coverage of the search space | Task 1 | ☐ |
| The expected color of each RYB combination | Task 2 | ☐ |
| Python code that collects the OT-2 data | Task 3 | ☐ |
| An evaluation of the DoE, and the response surface | Tasks 4, 6, 7 | ☐ |
| Random errors and systematic errors, with a discussion | Task 8, §8.2 | ☐ |
| A three-factor factorial ANOVA, with all effects | Tasks 5, 6, §8.1 | ☐ |
| Methods to decrease the errors, at both stages | §8.3 | ☐ |
| **Figure 1** — ternary diagram of the sample positions | Task 7 | ☐ |
| **Figure 2** — ternary response surface | Task 7 | ☐ |
| **Figure 3** — expected color against measured color | Task 9 | ☐ |
| **Figure 4** — normal probability plot of the residuals | Task 8 | ☐ |
| **Table** — the ANOVA table | Task 6 | ☐ |
| Markdown text: introduction, analysis, and summary | Part 0, §8.4 | ☐ |
| The random seed is 403 or 1003 | `SEED`, in Part 1 | ☐ |
| A CSV file with the RYB inputs and the 8 responses | `doe_submission.csv` | ☐ |

Save your data before you close the notebook. Assignment #3 uses it again.

In [ ]:
# Write the data for the submission, and for Assignment #3.
export_columns = (["run_order", "block", "R", "Y", "B", "well_index", "timestamp"]
                  + CHANNELS + ["A_total", "rmse_pred"])
export = results[[c for c in export_columns if c in results.columns]]
export.to_csv("doe_submission.csv", index=False)
print(f"doe_submission.csv: {len(export)} experiments, {len(export.columns)} columns")
export.head()

---

## What to do next

**Optimization is a different task.** You made a map of the design space with a
fixed design. [`BO.ipynb`](./BO.ipynb) in this folder uses Bayesian optimization
on the same equipment, and it searches for one target color. That algorithm
selects each experiment from all the earlier data.

Compare the two methods. DoE answers this question: how does this system
operate? Bayesian optimization answers a different question: what is the best
recipe? If you use your budget on the wrong method, you get a good answer to a
question that you did not ask. The [`bayesian_opt/`](../bayesian_opt/) workshop
gives more data about the second method.

**More factors than budget.** A full factorial design becomes large with 5 or
more factors (2⁵ = 32, and 2⁷ = 128). A **fractional factorial** design gives
most of the data with fewer experiments. In this design, the high-order
interactions are combined with the main effects. You accept this because you
expect those interactions to be very small. A **Plackett-Burman** design uses
fewer experiments again, for a first screen only.

**Curvature and optimization.** If your center points show significant
curvature, the next step is a **central composite design**. This design adds
axial points to the factorial design that you already did, and it fits a
quadratic response surface. Your 16 experiments stay in the analysis. This is
one more reason to design the full campaign before you start.

### References

> Baird, S. G.; Sparks, T. D. What Is a Minimal Working Example for a
> Self-Driving Laboratory? *Matter* **2022**, 5 (12), 4170–4178.
> https://doi.org/10.1016/j.matt.2022.11.007

> Baird, S. G.; Sparks, T. D. Building a "Hello World" for Self-Driving Labs:
> The Closed-Loop Spectroscopy Lab Light-Mixing Demo. *STAR Protocols* **2023**,
> 4 (2), 102329. https://doi.org/10.1016/j.xpro.2023.102329

- Montgomery, D. C. *Design and Analysis of Experiments*. This is the standard
  text. Chapters 5 and 6 give the theory of factorial designs and 2ᵏ analysis.
- [NIST/SEMATECH e-Handbook of Statistical Methods, chapter 5](https://www.itl.nist.gov/div898/handbook/pri/pri.htm).
  This handbook is free, and it covers DoE fully.
- [`self-driving-lab-demo`](https://github.com/sparks-baird/self-driving-lab-demo).
  This is the open-source demo that gives the background in Part 0.
- [Acceleration Consortium](https://acceleration.utoronto.ca/maps). This group
  operates the OT-2 platform that you used.

---

## Appendix — Solutions

Do the task before you read the answer. You learn when you are stuck, and this
appendix removes that step.

Each solution below replaces the full code of one task cell.

<details>
<summary><b>Solution — Task 1 — Make the design matrix</b></summary>

```python
LOW = 30.0
HIGH = 90.0
CENTER = 60.0

N_REPLICATES = 2
N_CENTER = 4

corner_rows = []
for _replicate in range(N_REPLICATES):
    for r_vol, y_vol, b_vol in itertools.product([LOW, HIGH], repeat=3):
        corner_rows.append({"block": "factorial", "R": r_vol, "Y": y_vol, "B": b_vol})

center_rows = [{"block": "center", "R": CENTER, "Y": CENTER, "B": CENTER}
               for _ in range(N_CENTER)]

VALIDATION_FRACTIONS = [
    (0.25, 0.75, 0.50), (0.75, 0.25, 0.00), (0.00, 0.50, 1.00),
    (1.00, 0.00, 0.25), (0.50, 1.00, 0.75),
]
validation_rows = [
    {"block": "validation",
     "R": LOW + fr * (HIGH - LOW), "Y": LOW + fy * (HIGH - LOW), "B": LOW + fb * (HIGH - LOW)}
    for fr, fy, fb in VALIDATION_FRACTIONS
]

design = pd.DataFrame(corner_rows + center_rows + validation_rows)
design = design.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
design.insert(0, "run_order", np.arange(1, len(design) + 1))
design["well_index"] = design["run_order"] + 1   # the pipeline test uses well 1

design
```

</details>

<details>
<summary><b>Solution — Task 2 — Calculate the expected color</b></summary>

```python
def predict_absorbance(r_vol, y_vol, b_vol):
    """Calculate the Beer-Lambert absorbance of the mixture in each channel."""
    total = r_vol + y_vol + b_vol
    fractions = {"R": r_vol / total, "Y": y_vol / total, "B": b_vol / total}
    return PATHLENGTH_CM * sum(DYE_ABSORPTIVITY[dye] * fractions[dye] for dye in fractions)


def predict_spectrum(r_vol, y_vol, b_vol):
    """Calculate the expected sensor counts for the mixture."""
    return REFERENCE_SPECTRUM * 10.0 ** (-predict_absorbance(r_vol, y_vol, b_vol))


print("A(60, 60, 60) =", np.round(predict_absorbance(60, 60, 60), 3))
print("I(60, 60, 60) =", np.round(predict_spectrum(60, 60, 60), 1))
```

</details>

<details>
<summary><b>Solution — Task 3 — Do the design</b></summary>

```python
records = []

for row in design.itertuples():
    spectrum, killed = run_experiment(row.R, row.Y, row.B, well_index=row.well_index)

    record = {
        "run_order": int(row.run_order),
        "block": row.block,
        "R": float(row.R), "Y": float(row.Y), "B": float(row.B),
        "well_index": int(row.well_index),
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    for channel, value in zip(CHANNELS, spectrum):
        record[channel] = value

    append_result(record)          # write to the disk first, every time
    records.append(record)
    print(f"experiment {row.run_order:2d}/{len(design)}  "
          f"R={row.R:5.1f} Y={row.Y:5.1f} B={row.B:5.1f}  ->  {spectrum_to_hex(spectrum)}")

    if killed:
        print("The queue stopped. All the data up to here is on the disk.")
        break

results = pd.DataFrame(records)
print(f"\n{len(results)} experiments are complete")
```

</details>

<details>
<summary><b>Solution — Task 4 — Make the response variables</b></summary>

```python
def absorbance(spectrum, reference=REFERENCE_SPECTRUM):
    """Calculate the Beer-Lambert absorbance in each channel, from the counts."""
    spectrum = np.clip(np.asarray(spectrum, dtype=float), 1.0, None)
    return -np.log10(spectrum / reference)


# The absorbance in each channel
A = np.array([absorbance(row) for row in results[CHANNELS].to_numpy()])
for i, channel in enumerate(CHANNELS):
    results["A_" + channel] = A[:, i]

# The primary response: the total absorbance across the spectrum
results["A_total"] = A.sum(axis=1)

# The distance from each measurement to the prediction in Task 2
results["rmse_pred"] = [
    float(np.sqrt(np.mean(
        (np.array([getattr(row, ch) for ch in CHANNELS])
         - predict_spectrum(row.R, row.Y, row.B)) ** 2)))
    for row in results.itertuples()
]

results[["run_order", "block", "R", "Y", "B", "A_total", "rmse_pred"]]
```

</details>

<details>
<summary><b>Solution — Task 5 — Calculate the effects manually</b></summary>

```python
RESPONSE = "A_total"
FACTORS = ("R", "Y", "B")

factorial = results[results.block == "factorial"].copy()
half_range = (HIGH - LOW) / 2

# Step 1: code the factors to -1 and +1
for f in FACTORS:
    factorial["x" + f] = (factorial[f] - CENTER) / half_range

y = factorial[RESPONSE]
n_runs = len(factorial)
effects = {}

# Step 2: the main effects, as a difference of the means
for f in FACTORS:
    high_mean = y[factorial["x" + f] > 0].mean()
    low_mean = y[factorial["x" + f] < 0].mean()
    effects[f] = high_mean - low_mean

# Step 3: the interactions, from the product columns
for a, b in itertools.combinations(FACTORS, 2):
    factorial[f"x{a}{b}"] = factorial["x" + a] * factorial["x" + b]
    effects[a + b] = (factorial[f"x{a}{b}"] * y).sum() / (n_runs / 2)

factorial["xRYB"] = factorial["xR"] * factorial["xY"] * factorial["xB"]
effects["RYB"] = (factorial["xRYB"] * y).sum() / (n_runs / 2)

for name, value in effects.items():
    print(f"  {name:>4s}  {value:+.4f}")
```

</details>

<details>
<summary><b>Solution — Task 6 — ANOVA</b></summary>

```python
import statsmodels.api as sm
import statsmodels.formula.api as smf

formula = f"{RESPONSE} ~ xR * xY * xB"
model = smf.ols(formula, data=factorial).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print(f"residual standard deviation: {np.sqrt(model.mse_resid):.4f}")
print(f"R-squared: {model.rsquared:.4f}\n")
anova_table.round(5)
```

</details>

<details>
<summary><b>Solution — Task 7 — Ternary diagrams</b></summary>

```python
import plotly.express as px

total_volume = results.R + results.Y + results.B
results["fR"] = results.R / total_volume
results["fY"] = results.Y / total_volume
results["fB"] = results.B / total_volume
results["total_volume"] = total_volume

# Figure 1 — the sample positions
fig1 = px.scatter_ternary(
    results, a="fR", b="fY", c="fB",
    color="block",
    hover_data=["run_order", "R", "Y", "B", "total_volume"],
    title="Figure 1 - DoE sample positions",
)
fig1.update_traces(marker=dict(size=12, line=dict(width=1, color="white")))
fig1.show()

# Figure 2 — the response surface
fig2 = px.scatter_ternary(
    results, a="fR", b="fY", c="fB",
    color=RESPONSE,
    color_continuous_scale="Viridis",
    hover_data=["run_order", "R", "Y", "B", "total_volume"],
    title=f"Figure 2 - response surface ({RESPONSE})",
)
fig2.update_traces(marker=dict(size=14, line=dict(width=1, color="white")))
fig2.show()
```

</details>

<details>
<summary><b>Solution — Task 8 — Examine the residuals</b></summary>

```python
from scipy import stats

factorial["fitted"] = model.fittedvalues
factorial["residual"] = model.resid
factorial["plate_col"] = [plate_position(w)[1] for w in factorial.well_index]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

# (a) Figure 4 — the normal probability plot of the residuals
stats.probplot(factorial.residual, dist="norm", plot=axes[0])
axes[0].set_title("Figure 4 - normal probability plot")

# (b) residual against run order — this finds a drift during the session
axes[1].scatter(factorial.run_order, factorial.residual, color="#4c72b0")
axes[1].axhline(0, ls=":", c="grey")
axes[1].set_xlabel("run order")
axes[1].set_ylabel("residual")
axes[1].set_title("Residuals against run order")

# (c) residual against plate column — this finds a position effect
axes[2].scatter(factorial.plate_col, factorial.residual, color="#dd8452")
axes[2].axhline(0, ls=":", c="grey")
axes[2].set_xlabel("plate column")
axes[2].set_title("Residuals against plate column")

fig.tight_layout()
plt.show()

r_pos, p_pos = stats.pearsonr(factorial.plate_col, factorial.residual)
r_time, p_time = stats.pearsonr(factorial.run_order, factorial.residual)
print(f"residual against plate column : r = {r_pos:+.3f}  p = {p_pos:.4f}")
print(f"residual against run order    : r = {r_time:+.3f}  p = {p_time:.4f}")
```

</details>

<details>
<summary><b>Solution — Task 9 — Expected color against measured color</b></summary>

```python
validation = results[results.block == "validation"].copy()

predicted = np.array([predict_spectrum(row.R, row.Y, row.B)
                      for row in validation.itertuples()])
measured = validation[CHANNELS].to_numpy()

# Figure 3 — the parity plot
fig, ax = plt.subplots(figsize=(4.8, 4.8))
ax.scatter(predicted.ravel(), measured.ravel(), alpha=0.75, color="#4c72b0")
lims = [min(predicted.min(), measured.min()) * 0.95,
        max(predicted.max(), measured.max()) * 1.05]
ax.plot(lims, lims, "k--", lw=1, label="correct prediction")
ax.set_xlabel("expected counts")
ax.set_ylabel("measured counts")
ax.set_title("Figure 3 - expected against measured")
ax.legend()
fig.tight_layout()
plt.show()

bias = float(np.mean(measured - predicted))
print(f"mean signed error (measured - expected): {bias:+.1f} counts")
print(f"RMSE: {np.sqrt(np.mean((measured - predicted) ** 2)):.1f} counts")

# The color comparison
labels, hexes = [], []
for row, pred_spec in zip(validation.itertuples(), predicted):
    labels += [f"#{row.run_order} expect", f"#{row.run_order} measur"]
    hexes += [spectrum_to_hex(pred_spec),
              spectrum_to_hex(np.array([getattr(row, ch) for ch in CHANNELS]))]
show_swatches(labels, hexes)
```

</details>

<details>
<summary><b>Solution — Task 10 (optional) — Calibrate the dyes with your own data</b></summary>

```python
train = results[results.block == "factorial"]
C = train[["fR", "fY", "fB"]].to_numpy()
A_obs = train[["A_" + ch for ch in CHANNELS]].to_numpy()

E_fitted, *_ = np.linalg.lstsq(C, A_obs, rcond=None)

print("fitted absorptivities (rows: R, Y, B. columns: the 8 channels)")
print(np.round(E_fitted, 3))

# Predict the validation experiments again, with the new absorptivities
C_val = validation[["fR", "fY", "fB"]].to_numpy()
predicted_fitted = REFERENCE_SPECTRUM * 10.0 ** (-(C_val @ E_fitted))

rmse_before = np.sqrt(np.mean((measured - predicted) ** 2))
rmse_after = np.sqrt(np.mean((measured - predicted_fitted) ** 2))
print(f"\nvalidation RMSE, given absorptivities : {rmse_before:8.1f} counts")
print(f"validation RMSE, fitted absorptivities: {rmse_after:8.1f} counts")
```

</details>